# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charanya207/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a simple baseline model because it is easy to interpret and compare with the Week-4 baseline. It fits this lane because the goal is to predict the target consistently while using the same data and evaluation setup.

In [32]:
print("Chosen method: simple baseline models")
print("Reason: easy to interpret and compare with the Week-4 baseline.")

Chosen method: simple baseline models
Reason: easy to interpret and compare with the Week-4 baseline.


In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-based holdout split, keeping some clients completely in the test set. This is an honest split because pages from the same client do not appear in both training and test data, reducing the chance of the model learning client-specific patterns.

In [34]:
print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())
print("Client overlap:", len(set(train_df["client_id"]) & set(test_df["client_id"])))# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Training clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [35]:
import os

print("Current folder:", os.getcwd())

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

Current folder: /content
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv


In [36]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    print(root)
    for file in files:
        print("   ", file)

/content/flyrank-ml-internship
    requirements.txt
    GUIDE.md
    AGENTS.md
    CLAUDE.md
    README.md
    LICENSE
    .gitignore
    DATA_USE.md
    SETUP.md
/content/flyrank-ml-internship/scripts
    02_baseline_score.py
    01_prepare_features.py
    ml_utils.py
    04_evaluate_and_export.py
    run_all.py
    05_build_pdf_report.py
    03_train_model.py
/content/flyrank-ml-internship/data
/content/flyrank-ml-internship/data/raw
    content_refresh_anonymized.csv
/content/flyrank-ml-internship/work
    capstone_report_template.md
    README.md
/content/flyrank-ml-internship/work/notebooks
    w03_feature_leakage_check.ipynb
    w01_research_question.ipynb
    w02_ml_task_framing.ipynb
    capstone.ipynb
    w03_data_contract.ipynb
    w06_validation_audit.ipynb
    w04_baseline_score.ipynb
    w07_action_playbook.ipynb
    w05_model.ipynb
    w04_signal_audit.ipynb
/content/flyrank-ml-internship/.github
/content/flyrank-ml-internship/.github/workflows
    personalize.yml
    dat

In [37]:
import os

print("Checking dataset...")

possible_paths = [
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv"
]

for path in possible_paths:
    print(path, "->", os.path.exists(path))

Checking dataset...
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv -> True
/content/data/raw/content_refresh_anonymized.csv -> False
data/raw/content_refresh_anonymized.csv -> False


In [38]:
!cd /content/flyrank-ml-internship && git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [39]:
!cd /content/flyrank-ml-internship && git ls-files data/raw/

data/raw/content_refresh_anonymized.csv


In [40]:
!git clone https://github.com/Charanya207/flyrank-ml-internship.git /content/flyrank-ml-internship

fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.


In [41]:
import os

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(path))

Dataset exists: True


In [42]:
import pandas as pd

# Load the dataset
df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [43]:
# Create the target label
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nClients:", df["client_id"].nunique())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Clients: 32


In [44]:
from sklearn.model_selection import train_test_split

# Get unique clients
clients = df["client_id"].dropna().unique()

# Hold out 20% of clients for testing
train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

# Create train and test sets
train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

# Verify no client appears in both sets
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Client overlap:", len(overlap))

Training clients: 25
Test clients: 7
Training rows: 26581
Test rows: 3419
Client overlap: 0


In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Columns that must NOT be used as features
leakage_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

# Use the remaining columns as features
feature_cols = [c for c in df.columns if c not in leakage_cols]

X_train = train_df[feature_cols]
y_train = train_df["is_declining_label"]

X_test = test_df[feature_cols]
y_test = test_df["is_declining_label"]

# Identify numerical and categorical columns
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

print("Features:", len(feature_cols))
print("Numeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))

Features: 40
Numeric features: 29
Categorical features: 11


In [46]:
# Train and compare models

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results)

print("Model comparison:")
display(results_df.round(4))

Model comparison:


,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.6832,0.6640,0.8001,0.7258
1,Decision Tree,0.7918,0.7217,0.9805,0.8314
2,Random Forest,0.7795,0.7081,0.9849,0.8239


I compare the learned models with my Week-4 baseline using the same data, metric, and client-holdout split. This makes the comparison fair and shows whether the trained model improves the baseline.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Decision Tree performed best overall based on the F1 score of 0.8314 and accuracy of 0.7918. Its recall of 0.9805 means it identifies most of the declining cases, but its precision of 0.7217 shows that some predicted declining cases are not actually declining.

The model can be wrong when the available content and traffic signals do not clearly indicate decline. The model may also rely strongly on traffic, engagement, freshness, and search-related signals. These relationships should be treated as decision-support signals rather than proof of causation.

In [47]:
# Error analysis for the best model: Decision Tree

best_model = models["Decision Tree"]

best_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

best_pipe.fit(X_train, y_train)
test_pred = best_pipe.predict(X_test)

# Show the main error counts
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, test_pred)

print("Confusion matrix:")
print(cm)

print("\nFalse positives:", cm[0, 1])
print("False negatives:", cm[1, 0])# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Confusion matrix:
[[ 951  677]
 [  35 1756]]

False positives: 677
False negatives: 35


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.